# Instructions

### Notebook needs to be migrated to Gemini.

# Workshop: Multi-Agent LLMs for Finance

In this example, we demonstrate a multi-agent system using large language models (LLMs) applied to finance. The system is composed of three agents:

1. **Data Agent:** Fetches financial data (e.g., historical stock prices for a ticker such as AAPL) using the `yfinance` library.
2. **Analysis Agent:** Reviews the data by computing basic statistics and provides a neutral analysis using an LLM (or simulated response).
3. **Decision Agent:** Generates a trading recommendation (Buy, Hold, or Sell) based on the analysis.

This notebook includes both code and explanatory text. You may need to set your OpenAI API key if you wish to use real API calls.


In [ ]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import datetime
import openai  # Ensure the openai package is installed if you plan to use actual API calls

import os
from openai import OpenAI

# Set your OpenAI API key if you have one; otherwise, the agents will return simulated responses.
# Uncomment and set your API key below:
openai.api_key = ""

## Define Agents

### Define Agent Base **Class**

In [ ]:
# Define a base agent class that uses the OpenAI API if an API key is set; otherwise, it returns a simulated response.
class BaseAgent:
    def __init__(self, name, role):
        self.name = name
        self.role = role  # This describes the agent's responsibility

    def act(self, prompt):
        # If an API key is provided, make a call to the OpenAI API.
        if openai.api_key:
            client = OpenAI(
                api_key=openai.api_key
            )
            chat_completion = client.chat.completions.create(
                  messages=[
                      {"role": "system", "content": self.role},
                      {"role": "user", "content": prompt}
                  ],
                  temperature=0.7,
                  max_tokens=1500,
                model="gpt-4o",
            )
            return chat_completion.choices[0].message.content.strip()
        else:
            return f"[Simulated response by {self.name} for prompt: {prompt[:50]}...]"


### Define Data Agent

In [ ]:
# Data Agent: Fetches financial data using yfinance.
class DataAgent(BaseAgent):
    def fetch_data(self, ticker, period='1mo', interval='1d'):
        print(f"{self.name} is fetching data for {ticker} (Period: {period}, Interval: {interval}).")
        stock = yf.Ticker(ticker)
        df = stock.history(period=period, interval=interval)
        return df



### Define Analysis Agent

In [ ]:
# Analysis Agent: Analyzes the data by computing summary statistics and generating insights.
class AnalysisAgent(BaseAgent):
    def act(self, prompt):
        # Use GPT-4 if an API key is provided; otherwise, return a simulated response.
        if openai.api_key:
          client = OpenAI(
              api_key=openai.api_key
          )

          chat_completion = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": self.role},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=1500,
              model="gpt-4o",
          )
          return chat_completion.choices[0].message.content.strip()
        else:
            return f"[Simulated response by {self.name} for prompt: {prompt[:50]}...]"

    def analyze(self, data_df):
        # Compute summary statistics from the DataFrame.
        summary = data_df.describe().to_string()
        prompt = f"Analyze the following financial data summary and provide neutral insights:\n{summary}"
        return self.act(prompt)

### Define Decision Agent

In [ ]:
# Decision Agent: Provides a trading recommendation based on the analysis.
class DecisionAgent(BaseAgent):
    def recommend(self, analysis):
        prompt = f"Based on the following analysis, provide a simple trading recommendation (Buy, Hold, or Sell) with reasoning:\n{analysis}"
        return self.act(prompt)


## Simulate Agent Interaction

In [ ]:
# Create instances of the agents with specific roles.
data_agent = DataAgent("DataAgent", "You fetch and provide historical stock data.")
analysis_agent = AnalysisAgent("AnalysisAgent", "You analyze financial data and provide neutral insights.")
decision_agent = DecisionAgent("DecisionAgent", "You provide trading recommendations based on analysis.")

# Step 1: Data Agent fetches historical data for a chosen stock.
ticker = "AAPL"  # Example ticker for Apple Inc.
data_df = data_agent.fetch_data(ticker, period='12mo', interval='1d')
print("\nFetched Data (last 5 records):")
print(data_df.tail())

# Step 2: Analysis Agent analyzes the fetched data.
analysis = analysis_agent.analyze(data_df)
print("\nAnalysis:")
print(analysis)

# Step 3: Decision Agent provides a trading recommendation based on the analysis.
recommendation = decision_agent.recommend(analysis)
print("\nTrading Recommendation:")
print(recommendation)

DataAgent is fetching data for AAPL (Period: 12mo, Interval: 1d).

Fetched Data (last 5 records):
                                 Open        High         Low       Close  \
Date                                                                        
2025-02-14 00:00:00-05:00  241.250000  245.550003  240.990005  244.600006   
2025-02-18 00:00:00-05:00  244.149994  245.179993  241.839996  244.470001   
2025-02-19 00:00:00-05:00  244.660004  246.009995  243.160004  244.869995   
2025-02-20 00:00:00-05:00  244.940002  246.779999  244.289993  245.830002   
2025-02-21 00:00:00-05:00  245.949997  248.690002  245.220001  245.550003   

                             Volume  Dividends  Stock Splits  
Date                                                          
2025-02-14 00:00:00-05:00  40896200        0.0           0.0  
2025-02-18 00:00:00-05:00  48822500        0.0           0.0  
2025-02-19 00:00:00-05:00  32204200        0.0           0.0  
2025-02-20 00:00:00-05:00  32316900        0.0 

# Conclusion

In this workshop, we simulated a multi-agent system using Python. The **Data Agent** retrieves historical stock data (for example, from Apple Inc. over the past month), the **Analysis Agent** computes summary statistics and generates neutral insights, and the **Decision Agent** produces a trading recommendation based on the analysis.

You can extend and refine each agent's functionality by incorporating more detailed analysis or additional financial data sources. If you have an OpenAI API key, you can obtain real-time responses from a language model instead of the simulated outputs shown here.

# Extension

## Create Report

In [ ]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import datetime
import openai  # Ensure the openai package is installed if you plan to use API calls
import plotly.graph_objects as go
import plotly.io as pio

# Uncomment and set your API key below if available.
# openai.api_key = "your-openai-api-key"

# Function to create a Plotly line chart for the closing prices of the stock.
def create_stock_plot(data_df, ticker):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=data_df.index,
        y=data_df['Close'],
        mode='lines+markers',
        name='Close Price'
    ))
    fig.update_layout(
        title=f'{ticker} Stock Closing Prices (1 Month Daily Data)',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        template='plotly_white'
    )
    return fig

# Function to convert a DataFrame to an HTML table.
def dataframe_to_html(data_df):
    return data_df.to_html(classes="table table-striped", border=0)

def generate_html_report(data_df, analysis, recommendation, plot_fig, output_filename='financial_report.html'):
    # Convert the DataFrame to HTML.
    data_html = dataframe_to_html(data_df)
    # Convert the Plotly figure to an HTML div.
    plot_html = pio.to_html(plot_fig, full_html=False, include_plotlyjs='cdn')

    # Build the full HTML content.
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Financial Multi-Agent Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
            }}
            h1, h2 {{
                color: #333;
            }}
            .section {{
                margin-bottom: 40px;
            }}
            .table {{
                border-collapse: collapse;
                width: 100%;
            }}
            .table th, .table td {{
                border: 1px solid #ddd;
                padding: 8px;
            }}
            .table th {{
                background-color: #f2f2f2;
                text-align: left;
            }}
            pre {{
                background-color: #f8f8f8;
                padding: 10px;
                border: 1px solid #ddd;
                overflow: auto;
                white-space: pre-wrap;       /* Ensures text wraps */
                word-wrap: break-word;       /* Breaks long words */
            }}
        </style>
    </head>
    <body>
        <h1>Financial Multi-Agent Report</h1>

        <div class="section">
            <h2>Fetched Data (Last 5 Records)</h2>
            {data_html}
        </div>

        <div class="section">
            <h2>Stock Closing Prices Chart</h2>
            {plot_html}
        </div>

        <div class="section">
            <h2>Analysis</h2>
            <pre>{analysis}</pre>
        </div>

        <div class="section">
            <h2>Trading Recommendation</h2>
            <pre>{recommendation}</pre>
        </div>
    </body>
    </html>
    """

    # Write the report to an HTML file.
    with open(output_filename, 'w') as f:
        f.write(html_content)
    print(f"Report written to {output_filename}")
    # Step 6: Open the generated HTML report in the default web browser.
    import webbrowser
    webbrowser.open(output_filename)


In [ ]:
# Step 2: Create a Plotly chart for the fetched data.
stock_fig = create_stock_plot(data_df, ticker)

# Step 5: Generate an HTML report combining the data, chart, analysis, and recommendation.
output_filename = 'financial_report.html'
generate_html_report(data_df, analysis, recommendation, stock_fig, output_filename=output_filename)



Report written to financial_report.html


# Exercises

## 1. Change the stock to another one

## 2. Add a new agent, which takes trends into account.
Trend-following is a general strategy which follows upward or downward trends of prices.